# Stage 09 — 阶段 7：全量语料 live 评估

本 notebook **独立于** `answer-eval-cache-batch.ipynb`（阶段 0–6 样本库验证），用于在全量 PMC 检索链路上复跑评估。

| 对比项 | 样本 notebook（C0–C6） | 本 notebook（阶段 7） |
|--------|------------------------|------------------------|
| 检索语料 | 1,267 chunks（sample） | **6,107,296** chunks（full） |
| 运行模式 | 默认 offline 快照 | **live**（Ollama + 全量检索） |
| 产出报告 | `eval_cache_batch_report.json` | `eval_cache_batch_report_full.json` |

> **耗时提示**：BM25 全量索引首次构建可能数小时；单条 live query（检索+生成）约 30s–数分钟。请按 C0→C6 顺序执行，重跑 live 前确认 `RUN_LIVE_EVAL=True`。

## C0：路径引导与 ground truth

In [1]:
from pathlib import Path
import json
import sys

CWD = Path.cwd().resolve()
STAGE09_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SRC = STAGE09_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bootstrap import bootstrap_paths

PATHS = bootstrap_paths(STAGE09_ROOT)
GT_PATH = STAGE09_ROOT / "data" / "ground_truth.json"
gt_payload = json.loads(GT_PATH.read_text(encoding="utf-8"))
GT_BY_QUERY = {item["query"]: item for item in gt_payload.get("queries", [])}
QUERIES = [item["query"] for item in gt_payload.get("queries", [])]

print("stage09:", STAGE09_ROOT)
print("queries:", len(QUERIES))
for i, q in enumerate(QUERIES, start=1):
    print(f"  {i}. {q}")

stage09: D:\谷歌\09 生成答案评估，缓存策略与批量处理
queries: 4
  1. What is the treatment for MI?
  2. metformin cardiovascular effects
  3. papers on malaria after 2015
  4. warfarin atrial fibrillation elderly


## C1：全量语料资源检查（7.0）

等价于 `06/scripts/run_retrieval_eval.py --check-only --mode full`。

In [2]:
from full_eval import check_full_corpus_resources

resource_check = check_full_corpus_resources(PATHS["stage06"])
resource_check

{'mode': 'full',
 'chunks_path': 'D:\\谷歌\\09 生成答案评估，缓存策略与批量处理\\data\\oa_comm_chunks.jsonl',
 'chunks_exists': True,
 'slim_path': 'D:\\谷歌\\06 检索系统开发第二部分\\data\\oa_comm_slim.jsonl',
 'slim_exists': True,
 'chroma_persist_dir': 'D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db_full',
 'chroma_exists': True,
 'collection': 'pmc_oa_comm_full',
 'corpus_size_hint': 6107296,
 'ready': True,
 'status': 'ready'}

## C2：Ollama 服务探测

live 评估前须 `Probe OK`；若失败请先启动 Ollama 并拉取 `deepseek-r1:7b`。

In [3]:
from full_eval import probe_ollama

ollama_probe = probe_ollama(PATHS["stage08"])
ollama_probe

{'ok': True,
 'base_url': 'http://127.0.0.1:11434',
 'model_requested': 'deepseek-r1:7b',
 'model_available': True,
 'models_preview': ['deepseek-r1:7b']}

## C2.5：全量 BM25 离线索引（09 落盘 → `data/bm25_full`）

**数据路径（阶段 7 本地快路径）**：
- 全量语料：`09 .../data/oa_comm_chunks.jsonl`（已从 E: 迁入，BM25 构建时读取）
- BM25 索引：`09 .../data/bm25_full/`（构建产物）

**与 04 `chroma_db_full` 的区别**：
- `chroma_db_full` = **向量库**（04 阶段已建好，D: 工程内）
- `data/bm25_full/` = **BM25 关键词索引**（本 cell 构建）

构建一次后，06 `MultiPathRetriever.from_mode("full")` 会自动加载。E: 仅作手动备份，不参与自动读取。

In [ ]:
from bm25_store import build_sharded_full_bm25, resolve_bm25_full_cache_dir, sharded_status

# 分片构建：低内存、断点续建、逐片进度。死机时把 SHARD_SIZE 调更小（如 50000）。
RUN_BM25_BUILD = True   # 需要构建时改 True
SHARD_SIZE = 100_000     # 每片 chunk 数；越小内存峰值越低、片数越多
BM25_SMOKE_LIMIT = None  # smoke 可设 100000（只建前 N 条）

status = sharded_status(PATHS["stage06"])
print("sharded status:", status.get("completed"), "| shard_files:", status.get("num_shard_files"))
if status.get("progress"):
    p = status["progress"]
    print("  progress:", p.get("status"), "completed_shards=", p.get("completed_shards"),
          "valid_chunks=", p.get("valid_chunks"))

if RUN_BM25_BUILD:
    manifest = build_sharded_full_bm25(
        PATHS["stage06"],
        output_dir=resolve_bm25_full_cache_dir(),
        shard_size=SHARD_SIZE,
        resume=True,            # 中断后重跑本 cell 会从断点继续
        limit=BM25_SMOKE_LIMIT,
    )
    print("build result:", manifest.get("status"),
          "num_shards=", manifest.get("num_shards"),
          "total_chunks=", manifest.get("total_chunks"),
          "elapsed=", manifest.get("elapsed_seconds"), "s")
else:
    print("RUN_BM25_BUILD=False — 跳过构建。若已 completed，C5 将自动加载分片索引。")
    print("目标路径:", resolve_bm25_full_cache_dir())

sharded status: False | shard_files: 0
[bm25-shard] shard=1 chunks_total=100000 lines=100000 elapsed=10.9s
[bm25-shard] shard=2 chunks_total=200000 lines=200000 elapsed=21.4s
[bm25-shard] shard=3 chunks_total=300000 lines=300000 elapsed=32.3s
[bm25-shard] shard=4 chunks_total=400000 lines=400000 elapsed=43.8s
[bm25-shard] shard=5 chunks_total=500000 lines=500000 elapsed=55.3s
[bm25-shard] shard=6 chunks_total=600000 lines=600000 elapsed=66.9s
[bm25-shard] shard=7 chunks_total=700000 lines=700000 elapsed=78.3s
[bm25-shard] shard=8 chunks_total=800000 lines=800000 elapsed=89.7s
[bm25-shard] shard=9 chunks_total=900000 lines=900000 elapsed=101.1s
[bm25-shard] shard=10 chunks_total=1000000 lines=1000000 elapsed=112.5s
[bm25-shard] shard=11 chunks_total=1100000 lines=1100000 elapsed=123.9s
[bm25-shard] shard=12 chunks_total=1200000 lines=1200000 elapsed=135.3s
[bm25-shard] shard=13 chunks_total=1300000 lines=1300000 elapsed=146.7s
[bm25-shard] shard=14 chunks_total=1400000 lines=1400000 ela

## C3：06 全量检索快照（7.1）

> **归属说明**：`pipeline_eval_full.json` 由 **06 阶段** `run_retrieval_eval.py` 产出；本 notebook C3 **只负责触发 CLI 或检查文件是否已存在**，不是该 JSON 的「归属阶段」。

若文件不存在，可设 `RUN_RETRIEVAL_EVAL=True` 触发全量检索（**耗时长**）。

或在外部终端执行：
```powershell
cd "06 检索系统开发第二部分"
python scripts/run_retrieval_eval.py --mode full --queries "What is the treatment for MI?" "metformin cardiovascular effects" "papers on malaria after 2015" "warfarin atrial fibrillation elderly" --output outputs/samples/pipeline_eval_full.json
```

In [5]:
import json
import subprocess
import sys

RUN_RETRIEVAL_EVAL = False  # 首次全量检索时改为 True
SKIP_RERANK_FOR_SMOKE = False  # 快速 smoke 可 True；正式评估建议 False

PIPELINE_EVAL_SAMPLE = PATHS["stage06"] / "outputs" / "samples" / "pipeline_eval.json"
PIPELINE_EVAL_FULL = PATHS["stage06"] / "outputs" / "samples" / "pipeline_eval_full.json"

if RUN_RETRIEVAL_EVAL:
    cmd = [
        sys.executable,
        str(PATHS["stage06"] / "scripts" / "run_retrieval_eval.py"),
        "--mode", "full",
        "--output", str(PIPELINE_EVAL_FULL),
        "--queries",
        *QUERIES,
    ]
    if SKIP_RERANK_FOR_SMOKE:
        cmd.append("--skip-rerank")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PATHS["stage06"]))

{
    "sample_exists": PIPELINE_EVAL_SAMPLE.is_file(),
    "full_exists": PIPELINE_EVAL_FULL.is_file(),
    "full_path": str(PIPELINE_EVAL_FULL),
}

{'sample_exists': True,
 'full_exists': True,
 'full_path': 'D:\\谷歌\\06 检索系统开发第二部分\\outputs\\samples\\pipeline_eval_full.json'}

## C4：样本库 vs 全量库 Top-1 检索对比

观察全量切换后 reranked Top-1 `doc_id` 是否变化（尤其 Q1 MI、Q3 malaria 2015+）。

In [6]:
from full_eval import compare_pipeline_eval_snapshots, save_json

if PIPELINE_EVAL_FULL.is_file():
    pipeline_compare_rows = compare_pipeline_eval_snapshots(
        PIPELINE_EVAL_SAMPLE,
        PIPELINE_EVAL_FULL,
        queries=QUERIES,
    )
    compare_path = STAGE09_ROOT / "outputs" / "samples" / "pipeline_eval_sample_vs_full.json"
    save_json(compare_path, {"queries": pipeline_compare_rows})
    pipeline_compare_rows
else:
  print("Skip: pipeline_eval_full.json not found. Run C3 first.")

## C4.5：分片 BM25 vs 单体 BM25 — keyword top-10 重叠率

量化分片近似的实际影响，包含两组对照：

| 对照 | 含义 | 重叠率解读 |
|------|------|------------|
| **A. 同语料对照** | 样本库上：分片 BM25 vs 单体 BM25 | 隔离分片算法误差；接近 1.0 说明分片近似可接受 |
| **B. 工程切换** | 全量分片 BM25 vs 样本单体 BM25 | 语料规模不同，重叠率低是**预期**（看 keyword 候选变化幅度） |

需 C2.5 全量 BM25 已完成（B 组）；A 组仅依赖样本语料，秒级完成。

In [11]:
import importlib
import bm25_store
import full_eval

importlib.reload(bm25_store)  # 若上面 cell 已 import 过旧版本，这里强制刷新
importlib.reload(full_eval)

from bm25_store import resolve_bm25_full_cache_dir
from full_eval import compare_bm25_sharded_vs_mono, save_json

BM25_OVERLAP_TOP_K = 10
BM25_SAMPLE_SHARD_SIZE = 400  # 样本仅 1267 条，分 4 片即可观察分片效应
RUN_FULL_CORPUS_BM25_COMPARE = True  # B 组需扫描全量 ~61 片，约 5–10 分钟；仅看分片误差可 False

bm25_overlap = compare_bm25_sharded_vs_mono(
    PATHS["stage06"],
    QUERIES,
    top_k=BM25_OVERLAP_TOP_K,
    sample_shard_size=BM25_SAMPLE_SHARD_SIZE,
    full_cache_dir=resolve_bm25_full_cache_dir(),
    include_full=RUN_FULL_CORPUS_BM25_COMPARE,
)

overlap_path = STAGE09_ROOT / "outputs" / "samples" / "bm25_sharded_vs_mono_overlap.json"
save_json(overlap_path, bm25_overlap)

ctrl = bm25_overlap["controlled_same_corpus"]
full_cmp = bm25_overlap["full_sharded_vs_sample_mono"]

print("=== A. 同语料对照（分片近似误差）===")
print("corpus_chunks:", ctrl["summary"]["corpus_chunks"])
print("avg_overlap_rate:", ctrl["summary"]["avg_overlap_rate"],
      "| avg_jaccard:", ctrl["summary"]["avg_jaccard"])
for row in ctrl["per_query"]:
    print(f"  [{row['overlap_count']}/{BM25_OVERLAP_TOP_K}] {row['query'][:50]}")

print("\n=== B. 全量分片 vs 样本单体（工程切换）===")
if full_cmp["summary"].get("ready"):
    print("full_chunks:", full_cmp["summary"]["full_chunks"],
          "| num_shards:", full_cmp["summary"]["full_num_shards"])
    print("avg_overlap_rate:", full_cmp["summary"]["avg_overlap_rate"],
          "| avg_jaccard:", full_cmp["summary"]["avg_jaccard"])
    for row in full_cmp["per_query"]:
        print(f"  [{row['overlap_count']}/{BM25_OVERLAP_TOP_K}] {row['query'][:50]}")
else:
    print(full_cmp["summary"].get("message", "not ready"))

print("\nsaved:", overlap_path)
bm25_overlap

[bm25-shard] shard=1 chunks_total=400 lines=400 elapsed=0.0s
[bm25-shard] shard=2 chunks_total=800 lines=800 elapsed=0.1s
[bm25-shard] shard=3 chunks_total=1200 lines=1200 elapsed=0.1s
[bm25-shard] shard=4 chunks_total=1267 lines=1267 elapsed=0.1s
=== A. 同语料对照（分片近似误差）===
corpus_chunks: 1267
avg_overlap_rate: 0.95 | avg_jaccard: 0.9167
  [10/10] What is the treatment for MI?
  [10/10] metformin cardiovascular effects
  [8/10] papers on malaria after 2015
  [10/10] warfarin atrial fibrillation elderly

=== B. 全量分片 vs 样本单体（工程切换）===
full_chunks: 6107296 | num_shards: 62
avg_overlap_rate: 0.0 | avg_jaccard: 0.0
  [0/10] What is the treatment for MI?
  [0/10] metformin cardiovascular effects
  [0/10] papers on malaria after 2015
  [0/10] warfarin atrial fibrillation elderly

saved: D:\谷歌\09 生成答案评估，缓存策略与批量处理\outputs\samples\bm25_sharded_vs_mono_overlap.json


{'top_k': 10,
 'controlled_same_corpus': {'summary': {'corpus': 'sample',
   'corpus_chunks': 1267,
   'sample_shard_size': 400,
   'avg_overlap_rate': 0.95,
   'avg_jaccard': 0.9167},
  'per_query': [{'query': 'What is the treatment for MI?',
    'keyword_query': 'treatment mi myocardial infarction heart attack',
    'mono_top_k': ['PMC512285',
     'PMC406391_chunk2',
     'PMC524185',
     'PMC406391_chunk1',
     'PMC516786',
     'PMC521694',
     'PMC514499',
     'PMC512290',
     'PMC406391_chunk0',
     'PMC521693'],
    'sharded_top_k': ['PMC512285',
     'PMC406391_chunk2',
     'PMC406391_chunk1',
     'PMC524185',
     'PMC516786',
     'PMC521694',
     'PMC514499',
     'PMC406391_chunk0',
     'PMC521693',
     'PMC512290'],
    'top_k': 10,
    'overlap_count': 10,
    'overlap_rate': 1.0,
    'jaccard': 1.0,
    'only_in_a': [],
    'only_in_b': [],
    'shared': ['PMC406391_chunk0',
     'PMC406391_chunk1',
     'PMC406391_chunk2',
     'PMC512285',
     'PMC512290',

## C5：全量 live 评估 — 首轮 + 缓存复跑（7.4）

- 使用 `RetrievalPipeline.from_mode("full")` + Ollama live 生成
- `context_text` 来自真实组装结果（非 `ctx::query` 占位）
- 设 `RUN_LIVE_EVAL=True` 后执行；建议 `MAX_WORKERS=2` 避免 Ollama 排队超时

In [7]:
import time
from batch_runner import BatchRunner
from full_eval import (
    build_full_eval_report,
    build_pipeline_with_eval_live_full,
    load_ground_truth,
    run_live_full_eval_task,
    save_json,
)

RUN_LIVE_EVAL = True #确认 C1/C2 通过后改为 True
MAX_WORKERS = 2
TEMPERATURE = 0.2
SKIP_EVIDENCE_EVAL = True
SKIP_CRITICAL_REVIEW = True
SKIP_RERANK = False  # 正式评估 False；快速 smoke True
LLM_TIMEOUT = 180.0

first_pass = []
second_pass = []
batch_stats = {}
model_name = ""

if RUN_LIVE_EVAL:
    if not resource_check.get("ready"):
        raise RuntimeError("Full corpus resources not ready. Fix C1 first.")
    if not ollama_probe.get("ok"):
        raise RuntimeError("Ollama not reachable. Fix C2 first.")

    gt_by_query = load_ground_truth(STAGE09_ROOT)
    pipe_eval, gen_pipe, model_name = build_pipeline_with_eval_live_full(
        PATHS["stage06"],
        PATHS["stage07"],
        PATHS["stage08"],
        skip_evidence_eval=SKIP_EVIDENCE_EVAL,
        skip_critical_review=SKIP_CRITICAL_REVIEW,
        max_context_tokens=1200,
        timeout=LLM_TIMEOUT,
        skip_rerank=SKIP_RERANK,
    )

    def _task(query: str) -> dict:
        return run_live_full_eval_task(
            pipe_eval,
            gen_pipe,
            query,
            gt_by_query[query],
            use_cache=True,
            force_refresh=False,
            temperature=TEMPERATURE,
            model_name=model_name,
        )

    runner = BatchRunner(max_workers=MAX_WORKERS)
    print(f"[live-full] first pass started at {time.strftime('%H:%M:%S')}")
    first_pass = runner.run_batch(QUERIES, _task, max_workers=MAX_WORKERS)
    print(f"[live-full] second pass (cache warm-up) at {time.strftime('%H:%M:%S')}")
    second_pass = runner.run_batch(QUERIES, _task, max_workers=MAX_WORKERS)
    batch_stats = runner.summarize(second_pass).to_dict()

    report_full = build_full_eval_report(
        mode="live",
        config={
            "queries": QUERIES,
            "temperature": TEMPERATURE,
            "max_workers": MAX_WORKERS,
            "model_name": model_name,
            "skip_evidence_eval": SKIP_EVIDENCE_EVAL,
            "skip_critical_review": SKIP_CRITICAL_REVIEW,
            "skip_rerank": SKIP_RERANK,
        },
        first_pass=first_pass,
        second_pass=second_pass,
        batch_stats=batch_stats,
        retrieval_mode="full",
        eval_subset="full_corpus",
    )

    out_report = STAGE09_ROOT / "outputs" / "samples" / "eval_cache_batch_report_full.json"
    out_log = STAGE09_ROOT / "outputs" / "logs" / f"eval_cache_batch_full_{time.strftime('%Y%m%d_%H%M%S')}.json"
    save_json(out_report, report_full)
    save_json(out_log, report_full)

    summary = report_full["summary"]
    {
        "saved_report": str(out_report),
        "saved_log": str(out_log),
        "cache_hit_rate_first": summary["cache_first_pass"]["hit_rate"],
        "cache_hit_rate_second": summary["cache_second_pass"]["hit_rate"],
        "rouge1_avg": summary["evaluation_first_pass"]["rouge1_avg"],
        "key_info_recall_avg": summary["evaluation_first_pass"]["key_info_recall_avg"],
        "hallucination_risk_avg": summary["evaluation_first_pass"]["hallucination_risk_avg"],
    }
else:
    print("RUN_LIVE_EVAL=False — 跳过 live 全量评估。完成 C1/C2 后设 True 再运行本 cell。")

[live-full] first pass started at 16:54:59
[Reranker] 加载 BAAI/bge-reranker-base（首次运行需下载 ~1.1GB，请耐心等待）...
[Reranker] 就绪，device=cuda
[live-full] second pass (cache warm-up) at 17:11:47


## C6：样本库 vs 全量库评估指标对照（7.5）

对比 `eval_cache_batch_report.json`（offline 样本链路）与 `eval_cache_batch_report_full.json`（live 全量链路）。

In [8]:
from full_eval import compare_eval_reports, save_json

SAMPLE_REPORT = STAGE09_ROOT / "outputs" / "samples" / "eval_cache_batch_report.json"
FULL_REPORT = STAGE09_ROOT / "outputs" / "samples" / "eval_cache_batch_report_full.json"
COMPARE_OUT = STAGE09_ROOT / "outputs" / "samples" / "eval_sample_vs_full.json"

if SAMPLE_REPORT.is_file() and FULL_REPORT.is_file():
    comparison = compare_eval_reports(SAMPLE_REPORT, FULL_REPORT)
    save_json(COMPARE_OUT, comparison)
    {
        "saved_to": str(COMPARE_OUT),
        "summary_delta": comparison["summary"]["delta"],
        "per_query": comparison["per_query"],
    }
else:
    print("Need both reports:")
    print("  sample:", SAMPLE_REPORT, SAMPLE_REPORT.is_file())
    print("  full:  ", FULL_REPORT, FULL_REPORT.is_file())

## C7：逐条结果速览（可选）

全量 live 跑完后，查看每条 query 的答案摘要与 key_info 命中情况。

In [9]:
import json

if FULL_REPORT.is_file():
    payload = json.loads(FULL_REPORT.read_text(encoding="utf-8"))
    rows = []
    for item in payload.get("first_pass", []):
        ev = item.get("evaluation") or {}
        gen = item.get("generation") or {}
        answer = str(gen.get("answer", ""))
        rows.append(
            {
                "query": item.get("query"),
                "rouge1": (ev.get("rouge") or {}).get("rouge1"),
                "key_info_recall": ev.get("key_info_recall"),
                "key_info_missing": ev.get("key_info_missing"),
                "cache_hit": (item.get("cache") or {}).get("hit"),
                "latency_seconds": item.get("latency_seconds"),
                "answer_preview": answer[:200] + ("..." if len(answer) > 200 else ""),
            }
        )
    rows
else:
    print("Full report not found. Complete C5 with RUN_LIVE_EVAL=True first.")